<!-- notebook-header -->
<a id="top"></a>

# 4. Stacks, Queues & Linked Lists

*LeetCode Playbook · notebook 4 of 12* · The linear structures: what waits on a stack, what a monotonic stack resolves, and pointer rewiring.

**In this notebook:** [Stacks & Queues](#s07) · [Monotonic Stack](#s08) · [Linked Lists](#s10)

← [Two Pointers, Sliding Window & Strings](03_Two_Pointers_Sliding_Window_Strings.ipynb) · [All notebooks](01_Start_Here.ipynb#notebooks) · [A-Z problem finder](12_Interview_Day.ipynb#finder) · [Binary Search & Sorting](05_Binary_Search_Sorting.ipynb) →

Run the setup cell below first; after that every section runs from its first cell.

In [ ]:
# Setup: everything this notebook imports. Run this cell first.
from collections import Counter, defaultdict, deque, OrderedDict
from functools import cmp_to_key, lru_cache
from itertools import accumulate, combinations, permutations, product
import bisect, heapq, math, random, string

<a id="s07"></a>

## Stacks & Queues

> A **stack** is a pile of unfinished business: whatever you opened last must be finished first, so it sits on top (LIFO). A **queue** is a line: whoever arrived first is served, or expires, first (FIFO). To pick one, ask a single question: *in what order do my pending items get finished?*

**Reach for it when** you see nesting or matching (brackets, `k[...]`, parentheses in an expression), "the most recent unmatched thing", going back (`..` in a path, undo), evaluating an expression, or a newcomer that only ever meets the nearest survivor (collisions): that is a **stack**. When items are handled in arrival order (the oldest expires first, players take turns, a fixed-size buffer), that is a **queue**, written with `collections.deque`. And the most common queue of all is BFS: "nearest first", "fewest steps" or "level by level" means the frontier is a deque (see [Graphs I](09_Graphs.ipynb#s17)).

**In this repo:** `stack/` (10 of its 16 problems; the "next greater" family is in Monotonic Stack) · `queues/` (6 problems) · bank: `practice/simple/13_valid_parentheses.py`, `practice/simple/14_min_stack.py`, `practice/simple/15_evaluate_reverse_polish_notation.py` · basics in `practice/simple/basics/stacks/`: `01_array_stack_and_queue_via_two_stacks.py`, `02_simplify_unix_path.py`, `03_decode_string.py`, `04_basic_calculator_ii.py`, `05_asteroid_collision.py`

### The picture

```text
s = { [ ( ) ] ( }                        the stack after each character (top on the right)

    {   opener: push                     {
    [   opener: push                     { [
    (   opener: push                     { [ (
    )   top is ( : a match, pop          { [
    ]   top is [ : a match, pop          {
    (   opener: push                     { (
    }   top is ( but } needs {           invalid: the innermost open bracket is (
```

```text
stack (LIFO):  [ a  b  c  d ]  <- push and pop here, at the top (d is the most recent)
queue (FIFO):  append -> [ e  d  c  b  a ] -> popleft         (a is the oldest, it leaves first)
```

**Why it is fast:** the brute force for nesting finds an innermost pair like `()`, deletes it, and rescans from the start: O(n²). A stack holds exactly the unfinished items in the order they were opened, so each character is pushed once and popped at most once: O(n). Queues have the same story in time: `list.pop(0)` shifts every remaining item (O(n) per call), while a `deque`, or a ring buffer that moves an index instead of the data, does it in O(1).

**Why it is correct:** brackets nest, so a bracket opened later must be closed earlier. The next closer can therefore only match the innermost open bracket, which is the most recent one: the top. A closer that doesn't match the top can never be matched by anything later, so failing at once is safe.

### From idea to code

**The idea in one sentence:** *walk the input once; push whatever is still unfinished; when an item finishes something, the thing it finishes is on top, so pop it and combine.*

The seven decisions, for stacks and queues side by side:

| Decision | Stack (nesting, evaluation) | Queue / deque (arrival order) |
|---|---|---|
| **State / Definition** | a list used as a stack of *unfinished* items, defined in a comment: `stack = openers not closed yet, innermost on top` | a `deque` in arrival order: `q = pings in [t - 3000, t], oldest at the front` |
| **Invariant** | after reading `s[:i]`, the stack holds exactly the unfinished items of that prefix, most recent on top | the deque holds exactly the live items, in arrival order |
| **Step** | an opener or a value: push it. A closer or an operator: pop what it finishes, combine, maybe push the result | `append` the new item at the back |
| **Fix** | when one newcomer can finish several items in a row (collisions, runs that vanish): `while` it beats the top, pop | `while` the front has expired: `popleft` |
| **Record** | when a pop measures something (a run length in 32; distances and areas in Monotonic Stack), record right there. Otherwise the answer is what is left at the end (`not stack`, the one value, the survivors). An early `return False` records a failure the moment it is certain | after the fix: `len(q)`, or the front |
| **Init** | `stack = []`, `num = 0`; sometimes a sentinel (a `-1` barrier, a trailing `"+"`) | `q = deque()` |
| **Return** | `not stack` (matching), the single value left (evaluation), the stack itself (survivors) | per call: the size or the front; at the end: whoever is left |

**What goes in one stack entry?** Exactly what you will need at the moment it is popped. It is the easiest decision to skip, and the one that makes the rest of the code obvious:

| Problem | One entry | Needed at the pop because |
|---|---|---|
| Valid Parentheses (20) | the opener | the closer must match its type |
| Evaluate RPN (150) | a value | an operator combines the two newest values |
| Basic Calculator II (227) | a signed term | `+` and `-` wait for the final sum; `*` and `/` edit the top |
| Decode String (394) | (text before `[`, count) | `]` glues prefix + inner × count |
| Calculator with parentheses (224, 772) | (terms so far, pending operator) | `)` resumes the whole outer expression |
| Longest Valid Parentheses (32) | an index, on a −1 barrier | a length is `i - stack[-1]` |
| Asteroid Collision (735) | a survivor | the newcomer fights the nearest survivor |
| Min Stack (155) | (value, min at or below it) | after a pop, the minimum must already be known |
| Remove Adjacent Duplicates II (1209) | [char, run length] | a run can grow again after the run above it vanishes |
| Simplify Path (71) | a directory name | `..` undoes the most recent one |

If you can't say what you'll need at the pop, solve two pops by hand first.

The words of the idea, line by line:

| In words | In code |
|---|---|
| "remember it, it's unfinished" | `stack.append(x)` |
| "the most recent unfinished thing" | `stack[-1]`, only after checking `stack` is not empty |
| "finish it" | `stack.pop()` |
| "anything still open?" | `return not stack` |
| "the right operand comes off first" | `b = stack.pop()`, then `a = stack.pop()`, then `a - b` |
| "read a number digit by digit" | `num = num * 10 + int(ch)` |
| "join the line" / "leave the line" | `q.append(x)` / `q.popleft()` (never `list.pop(0)`) |
| "the oldest item" | `q[0]` |

Two stack templates: one pops to *match*, the other pops to *combine*. Order matters in both: `is_valid` looks at the top *before* it pops (the check decides whether the pop is legal), and `eval_rpn` pops `b` before `a` (the right operand was pushed last).

In [ ]:
def is_valid(s):
    partner = {")": "(", "]": "[", "}": "{"}     # closer -> its opener
    stack = []                               # STATE + INIT: openers not closed yet, innermost on top
    for ch in s:
        if ch not in partner:
            stack.append(ch)                 # STEP (opener): one more unfinished bracket
        elif not stack or stack[-1] != partner[ch]:
            return False                     # RECORD: the answer is known now: invalid
        else:
            stack.pop()                      # STEP (closer): it finishes the innermost opener
    return not stack                         # RETURN: valid iff nothing was left open


OPS = {"+": lambda a, b: a + b, "-": lambda a, b: a - b,
       "*": lambda a, b: a * b, "/": lambda a, b: int(a / b)}   # int(a / b) truncates toward 0


def eval_rpn(tokens):
    stack = []                               # STATE + INIT: values waiting for an operator
    for tok in tokens:
        if tok in OPS:
            b = stack.pop()                  # the RIGHT operand was pushed last
            a = stack.pop()
            stack.append(OPS[tok](a, b))     # STEP (operator): two waiting values become one
        else:
            stack.append(int(tok))           # STEP (number): it waits; "-3" is a number too
    return stack[-1]                         # RETURN: exactly one value is left


print(is_valid("{[()]}"), is_valid("([)]"), is_valid("(("))                        # True False False
print(eval_rpn(["2", "1", "+", "3", "*"]), eval_rpn(["4", "13", "5", "/", "+"]))   # 9 6

**Try it**
- Delete `not stack or` from the `elif` and run `is_valid("]")`: `IndexError`, because the code peeks at the top of an empty stack.
- Replace `return not stack` with `return True` and run `is_valid("((")`: it says `True`, although nothing was ever closed.
- Swap the two pops (pop `a` first) and run `eval_rpn(["6", "3", "-"])`: you get -3 instead of 3, because the operands come off the stack in reverse order.
- Change `int(a / b)` to `a // b` in `OPS` and run `eval_rpn(["7", "-3", "/"])`: -3 instead of -2, because `//` floors and the problem truncates toward zero.

A queue template has the same shape, but items leave from the *other* end. Pings expire in the order they arrived, so only the front ever needs checking. The order is STEP, FIX, RECORD: appending first means the FIX loop can never empty the deque, because `t` itself never expires, so it needs no `self.q and` guard; measuring last means only live pings are counted.

In [ ]:
class RecentCounter:                         # 933: how many pings in [t - 3000, t]?
    def __init__(self):
        self.q = deque()                     # STATE + INIT: live pings, oldest at the front

    def ping(self, t):                       # t only grows
        self.q.append(t)                     # STEP: the newest joins at the back
        while self.q[0] < t - 3000:          # FIX: drop expired pings (q never empties: t is in it)
            self.q.popleft()
        return len(self.q)                   # RECORD + RETURN: the window is the deque


counter = RecentCounter()
print([counter.ping(t) for t in [1, 100, 3001, 3002]])   # [1, 2, 3, 3]

**Try it**
- Change `<` to `<=` in the `while` and replay `[1, 100, 3001, 3002]`: the third answer becomes 2, because the ping at `t = 1` is exactly 3000 ms old and still belongs to `[1, 3001]`.
- Change the `while` to `if` and ping `[1, 2, 3, 5000]`: the last answer is 3 instead of 1, since only one expired ping can leave per call.
- Replace `3000` with `10` and predict `[1, 5, 11, 12, 30]` before running: `[1, 2, 3, 3, 1]`.

### Watch it work

The stack after every token. Notice that `/` takes `13` and `5`, the two most recent values, and never touches the `4` underneath:

In [ ]:
def trace_rpn(tokens):
    stack = []
    for tok in tokens:
        note = ""
        if tok in OPS:
            b, a = stack.pop(), stack.pop()
            stack.append(OPS[tok](a, b))
            note = f"{a} {tok} {b} = {stack[-1]}"
        else:
            stack.append(int(tok))
        print(f"{tok:>3}   stack = {str(stack):<17} {note}".rstrip())


trace_rpn(["4", "13", "5", "/", "+"])

**Try it**
- Run `trace_rpn(["2", "1", "+", "3", "*"])` and predict the stack just before the `*` (it is `[3, 3]`).
- Trace `["10", "6", "9", "3", "+", "-11", "*", "/", "*", "17", "+", "5", "+"]`: the stack never holds more than 4 values, `6 / -132` truncates to 0, and the result is 22.
- Feed a broken expression, `trace_rpn(["1", "+"])`: the second `pop` raises `IndexError`. A defensive version checks `len(stack) >= 2` before popping.

### Where it goes wrong

1. **Peeking at an empty stack.** A closer with nothing open (`"]"`, `"())"`) makes `stack[-1]` raise `IndexError`. Check `not stack` first, in the same condition.
2. **Forgetting the leftovers.** `"(("` never fails inside the loop. The answer is `not stack`, not `True`.
3. **Counting instead of matching.** `"([)]"` has balanced counts but is invalid. A plain counter is enough only with a single bracket type (921): then every stack entry is the same `(` and only the height matters.
4. **Operand order.** The first pop is the *right* operand: `["6", "3", "-"]` is `6 - 3 = 3`, not -3.
5. **Floor instead of truncate.** Python's `//` floors (`7 // -3 == -3`), but these problems truncate toward zero (`int(7 / -3) == -2`). It also bites calculators that store `-3` as a signed term: `"14-3/2"` must be 13, and `-3 // 2` would make it 12.
6. **The last number is never applied.** A calculator applies an operator when the *next* operator arrives, so the final number needs a flush: loop over `s + "+"`. Without the sentinel, `"3+2*2"` gives 5. (Testing `i == len(s) - 1` also works, but only in an `if` of its own; chained as an `elif` after the digit test, `"42"` gives 0.)
7. **`if` instead of `while` for collisions.** A left-mover can destroy several right-movers in a row: `[1, 2, -5]` must end as `[-5]`, not `[1, -5]`.
8. **`list.pop(0)` as a queue.** Each call shifts every remaining item, so n dequeues cost about n²/2 moves. Use `deque.popleft()`, or a ring buffer that moves `head` instead of the data.
9. **An expiry loop without `q and`.** It is safe in `ping` only because `t` was just appended. In a read-only method such as `getHits(t)` (362, in [Design](11_Design.ipynb#s24)) write `while q and q[0] <= t - 300`, or a call on an empty counter raises `IndexError`.

### Edge cases to say out loud

Empty input · a lone closer · only openers · the wrong type innermost (`"([)]"`) · a single RPN token · negative numbers (`"-3"` is a number, not an operator) · dividing a negative · multi-digit numbers · a ping exactly 3000 ms old (still counts).

In [ ]:
assert is_valid("") is True
assert is_valid("]") is False                         # closer with nothing open
assert is_valid("((") is False                        # leftovers
assert is_valid("([)]") is False                      # counts match, order doesn't
assert eval_rpn(["5"]) == 5
assert eval_rpn(["7", "-3", "/"]) == -2               # "-3" is a number; truncate toward 0
assert eval_rpn(["6", "3", "-"]) == 3                 # a - b, with b popped first
assert eval_rpn(["10", "6", "9", "3", "+", "-11", "*", "/", "*", "17", "+", "5", "+"]) == 22
c = RecentCounter()
assert [c.ping(t) for t in [1, 3001, 3002]] == [1, 2, 2]   # t = 1 still counts at 3001
print("edge cases pass")

**Try it**
- Predict, then add: `assert is_valid("(]") is False` (the top is `(`, but `]` needs `[`).
- Write the assert for `eval_rpn(["3", "4", "-", "5", "*"])` before running it: `(3 - 4) * 5 = -5`.
- Swap the stack for one counter (+1 on an opener, −1 on a closer, fail below 0) and run `"([)]"`: it passes as valid. That is trap 3 in one line.

### Variations

| Variation | What changes from the template | Problems |
|---|---|---|
| **Match pairs** | push openers; a closer must equal the top | 20 |
| **Stack collapses to a counter** | one bracket type: keep only the height; a `)` at height 0 costs one insertion | 921 |
| **Index stack with a barrier** | push indices, seeded with `-1`; after a match the valid run is `i - stack[-1]` | 32 |
| **Postfix evaluation** | numbers push; an operator pops two and pushes one | 150 |
| **Precedence** | push signed terms; `*` and `/` edit the top term; the answer is `sum(stack)` | 227 |
| **Park and resume** | on an opener, push the current context and start fresh; on a closer, pop it and merge | 394, 224, 772 |
| **Park and resume, with time** | a start pauses the function below it; an end (inclusive) closes the top | 636 |
| **Collapse at the closer** | push everything; at `)` pop the group's operands, then its operator, and push one result | 1106 |
| **Score of nested pairs** | a stack of scores: `()` is 1, `(A)` doubles A, neighbours add up | 856 |
| **Mark, then rebuild** | a stack of indices of unmatched `(`; drop every marked character | 1249, in [Strings](03_Two_Pointers_Sliding_Window_Strings.ipynb#s20) |
| **Undo / go up** | names push, `..` pops (if anything is there), `.` and `""` do nothing | 71 |
| **Collisions** | the newcomer fights the top *while* they collide | 735 |
| **Stack of runs** | entry `[char, count]`; k in a row vanish, and the run below may grow again | 1209, 1047 |
| **Remember the minimum** | each entry stores `(value, min of everything at or below it)` | 155 |
| **Simulate the pops** | push in order; pop while the top equals the next value to pop | 946 |
| **Flatten nested lists** | a stack of iterators; the top is the list you are inside | 341 |
| **Queue from two stacks** | inbox takes pushes, outbox serves pops; refill only when the outbox is empty | 232 |
| **Stack from one queue** | after each push, rotate the older items behind the new one (`len - 1` times) | 225 |
| **Ring buffer** | fixed array + `head` + `count`; every index goes through `% k` | 622, 641 |
| **Time window** | deque of times; pop the front while it is too old | 933, 362 |
| **Round robin** | one queue per side; the earlier front acts and rejoins as `index + n` | 649 |

The first pass below covers the five shapes that come up most; the second pass (calculators with parentheses, fixed-size buffers, turn order) can wait until those feel easy.

**Precedence** (227): an operator can't be applied when you read it, because its right operand hasn't been read yet. So keep the operator *in front of* the current number in `op`, and apply it when the next operator arrives; the sentinel `"+"` at the end makes the last number get applied too. `+` and `-` are put off by pushing signed terms; `*` and `/` bind tighter, so they combine with the term on top right away.

In [ ]:
def push_term(stack, op, num):               # put num on the stack the way op says
    if op == "+":
        stack.append(num)
    elif op == "-":
        stack.append(-num)                   # the sign is stored with the term
    elif op == "*":
        stack.append(stack.pop() * num)      # binds to the term before it
    else:
        stack.append(int(stack.pop() / num)) # -3 / 2 -> -1 (truncate), not -2


def calculate_ii(s):
    stack, num, op = [], 0, "+"              # STATE + INIT: signed terms; op = the operator in front of num
    for ch in s + "+":                       # the sentinel "+" flushes the last number
        if ch.isdigit():
            num = num * 10 + int(ch)         # one more digit
        elif ch in "+-*/":
            push_term(stack, op, num)        # STEP: num is complete, apply the operator in front of it
            num, op = 0, ch
    return sum(stack)                        # RETURN: the postponed + and - happen here


print(calculate_ii("3+2*2"), calculate_ii(" 3/2 "), calculate_ii("14-3/2"))   # 7 1 13

**Try it**
- In `push_term`, change `int(stack.pop() / num)` to `stack.pop() // num` and run `calculate_ii("14-3/2")`: 12 instead of 13, because the term is stored as `-3` and `-3 // 2` floors to -2.
- Delete `+ "+"` from the loop and run `calculate_ii("3+2*2")`: 5 instead of 7. The pending `* 2` is never applied.
- Print `stack` just before the `return` for `"2*3+4*5-6/2"`: `[6, 20, -3]`, one signed term per `+` or `-`, which sum to 23.

**Park and resume** (394): an opening bracket means *save where I am and start fresh*; a closing bracket means *finish the inner part, restore what was saved, and merge*. Decode String parks `(text so far, repeat count)`; `cur` is always the text of the innermost open group.

In [ ]:
def decode_string(s):
    stack, cur, k = [], "", 0                # STATE + INIT: parked (text before '[', count); cur = open group's text
    for ch in s:
        if ch.isdigit():
            k = k * 10 + int(ch)             # "10[a]": counts can have several digits
        elif ch == "[":
            stack.append((cur, k))           # STEP: park, then start fresh inside
            cur, k = "", 0
        elif ch == "]":
            prev, count = stack.pop()        # STEP: resume the parked context
            cur = prev + cur * count
        else:
            cur += ch                        # STEP: a letter extends the open group
    return cur                               # RETURN


print(decode_string("3[a2[c]]"), decode_string("2[abc]3[cd]ef"))   # accaccacc abcabccdcdcdef

**Try it**
- Delete the line `cur, k = "", 0` and run `decode_string("a2[b]")`: `aabab` instead of `abb`. The parked prefix `"a"` was also left in the inner text, so it got repeated.
- Change `k = k * 10 + int(ch)` to `k = int(ch)` and run `decode_string("10[a]")`: an empty string, because only the last digit (0) survived.
- Print `stack` right after each push for `"3[a2[c]]"`: `[('', 3)]`, then `[('', 3), ('a', 2)]`. One frame per open bracket.
- Predict `decode_string("2[a]b")` before running: `aab`. Text after a group just extends `cur`.

**Collisions and undo** (735, 71): only a left-mover arriving after a right-mover can crash, and it meets the most recent survivor first. So it fights the top *while* it keeps winning (FIX), and it is pushed only if it survives (STEP). A path is a stack of directory names: a name goes one level down, `..` goes one level up, but never above the root (the same function is in `practice/simple/basics/stacks/02_simplify_unix_path.py`).

In [ ]:
def asteroid_collision(asteroids):
    stack = []                               # STATE + INIT: survivors so far, left to right
    for a in asteroids:
        alive = True
        while alive and a < 0 and stack and stack[-1] > 0:   # FIX: only -> <- collide
            if stack[-1] < -a:
                stack.pop()                  # the top explodes, a flies on
            else:
                if stack[-1] == -a:
                    stack.pop()              # same size: both explode
                alive = False
        if alive:
            stack.append(a)                  # STEP: a survives (for now)
    return stack                             # RETURN: the survivors


def simplify_path(path):
    stack = []                               # STATE + INIT: directory names, the root side at the bottom
    for part in path.split("/"):
        if part == "..":
            if stack:                        # never above the root
                stack.pop()                  # STEP: one level up
        elif part and part != ".":           # "" (from "//") and "." change nothing
            stack.append(part)               # STEP: one level down
    return "/" + "/".join(stack)             # RETURN


print(asteroid_collision([5, 10, -5]), asteroid_collision([8, -8]), asteroid_collision([1, 2, -5]))   # [5, 10] [] [-5]
print(simplify_path("/a/./b/../../c/"), simplify_path("/../"), simplify_path("/home//foo/"))   # /c / /home/foo

**Try it**
- Change the `while` in `asteroid_collision` to `if` and run `[1, 2, -5]`: `[1, -5]` instead of `[-5]`. One fight per asteroid is not enough.
- Delete the two lines of the same-size case and run `[8, -8]`: `[8]` instead of `[]`.
- Run `asteroid_collision([-2, -1, 1, 2])`: nothing collides. Left-movers already on the left never meet the right-movers behind them.
- Remove the `if stack:` guard and run `simplify_path("/../")`: `IndexError`, because `..` at the root tries to pop an empty stack.

**Entries that remember** (155, 1209): when the pop must know something about what lies *below* the top, store it in the entry. A stack only changes at its top, so the minimum of everything below an entry can't change while that entry is there: Min Stack stores it next to the value. Remove Adjacent Duplicates II stores a run as `[char, count]`, because when a run of k vanishes, the run underneath can keep growing.

In [ ]:
class MinStack:                              # 155
    def __init__(self):
        self.stack = []                      # STATE + INIT: (value, min of this entry and everything below)

    def push(self, x):
        low = min(x, self.stack[-1][1]) if self.stack else x
        self.stack.append((x, low))          # STEP: the entry carries its own minimum

    def pop(self):
        self.stack.pop()                     # the entry below still knows its own minimum

    def top(self):
        return self.stack[-1][0]

    def getMin(self):
        return self.stack[-1][1]


def remove_duplicates(s, k):                 # 1209
    stack = []                               # STATE + INIT: [char, how many in a row]
    for ch in s:
        if stack and stack[-1][0] == ch:
            stack[-1][1] += 1                # STEP: the run on top grows
            if stack[-1][1] == k:
                stack.pop()                  # FIX: k in a row vanish; the run below may grow again
        else:
            stack.append([ch, 1])            # STEP: a new run starts
    return "".join(ch * n for ch, n in stack)    # RETURN


ms = MinStack()
for x in [-2, 0, -3]:
    ms.push(x)
before = ms.getMin()
ms.pop()
print(before, ms.top(), ms.getMin())                                           # -3 0 -2
print(remove_duplicates("deeedbbcccbdaa", 3), remove_duplicates("abbaca", 2))   # aa ca

**Try it**
- Print `ms.stack` after the three pushes: `[(-2, -2), (0, -2), (-3, -3)]`. The right column is the running minimum, frozen at each height.
- Build the two-stack version instead: a `mins` stack pushed only when `x < mins[-1]`, popped when the popped value equals `mins[-1]`. Push 0, push 0, pop, then `getMin`: `IndexError`, because the single 0 in `mins` was shared by both pushes. Push when `x <= mins[-1]`.
- Print `stack` after every character of `"deeedbbcccbdaa"` with k = 3: when `ccc` vanishes, the `bb` underneath meets one more `b` and vanishes too, and then `ddd` does the same.
- Make the entry a tuple, `(ch, 1)`: `TypeError` on `stack[-1][1] += 1`, because a tuple can't change. That is why the entry is a list.

**A queue from two stacks** (232): pouring one stack into another reverses it, which turns "newest on top" into "oldest on top". Pour only when the outbox is empty, so every item is poured once: amortized O(1).

In [ ]:
class MyQueue:                               # 232
    def __init__(self):
        self.inbox, self.outbox = [], []     # STATE + INIT: inbox newest on top; outbox oldest on top

    def push(self, x):
        self.inbox.append(x)                 # STEP

    def peek(self):
        if not self.outbox:                  # FIX: refill only when empty, or the order breaks
            while self.inbox:
                self.outbox.append(self.inbox.pop())
        return self.outbox[-1]               # RETURN: the oldest item

    def pop(self):
        self.peek()                          # make sure the oldest item is on the outbox top
        return self.outbox.pop()

    def empty(self):
        return not self.inbox and not self.outbox


q = MyQueue()
q.push(1)
q.push(2)
first = q.pop()
q.push(3)                                    # 3 must wait behind 2
print(first, q.pop(), q.pop(), q.empty())    # 1 2 3 True

**Try it**
- In `peek`, remove `if not self.outbox:` so it always pours, and replay the lines above: the second `pop` returns 3 instead of 2, because 3 was poured on top of the waiting 2.
- Print `q.inbox, q.outbox` after every call: each item crosses from inbox to outbox exactly once.
- Push 1 to 1000, then pop all 1000: count the appends to `outbox`. There are exactly 1000, so the expensive pour is paid once per item.

**Second pass.** A calculator with parentheses parks `(terms so far, pending operator)` at each `(`, the same move as Decode String, and a finished group becomes one number for the outer expression. (For 224 alone there is a shortcut: with only `+` and `-`, a parenthesis just flips signs, so a stack of signs is enough.)

In [ ]:
def calculate(s):                            # + - * / and parentheses (224, 227, 772)
    stack, num, op, saved = [], 0, "+", []   # STATE + INIT: saved = (terms, op) parked at each "("
    for ch in s + "+":                       # the sentinel "+" flushes the last number
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch == "(":
            saved.append((stack, op))        # STEP: park the outer expression, start fresh
            stack, op = [], "+"
        elif ch in "+-*/)":
            push_term(stack, op, num)        # STEP: num is complete, apply the operator in front of it
            if ch == ")":                    # the group is finished: it becomes one number
                num = sum(stack)
                stack, op = saved.pop()      # resume the outer expression
            else:
                num, op = 0, ch
    return sum(stack)                        # RETURN


print(calculate("(1+(4+5+2)-3)+(6+8)"), calculate("-(2-3)"), calculate("2*(5+5*2)/3"))   # 23 1 10

**Try it**
- Print `saved` right after it grows, for `calculate("2*(3+(4-1))")`: first `[([2], '*')]`, then `[([2], '*'), ([3], '+')]`. Each `(` parks one more outer expression. The answer is 12.
- Delete `num = sum(stack)` and run `calculate("2*(3+4)")`: 8 instead of 14. The outer `*` multiplied only the group's last number.
- Predict `calculate("-(3*(2-5))")` before running: `-(3 * -3) = 9`.

**Ring buffer** (622, 641): a bounded queue in a fixed array. Dequeuing shouldn't move the data, only the label "this slot is the front". Bend the array into a ring: the items are `count` slots in a row starting at `head`, and every index goes through `% k`, so the line can wrap past the last slot.

```text
k = 5, head = 3, count = 3        slot:   0    1    2    3    4
                                  item:   c    .    .    a    b      front = buf[3] = a
                                                                     rear  = buf[(3 + 3 - 1) % 5] = buf[0] = c
the next enQueue writes buf[(3 + 3) % 5] = buf[1]
```

The deque version (641) adds two moves: `insertFront` steps back *first*, `head = (head - 1) % k`, then writes `buf[head]`; `deleteLast` is just `count -= 1`. Python's `%` never returns a negative number here; in Java or C++ write `(head - 1 + k) % k`.

In [ ]:
class MyCircularQueue:                       # 622 (641 adds operations at the front)
    def __init__(self, k):
        self.buf, self.k = [0] * k, k
        self.head, self.count = 0, 0         # STATE + INIT: items are buf[head], buf[head + 1], ... (mod k)

    def enQueue(self, x):
        if self.count == self.k:
            return False                     # full
        self.buf[(self.head + self.count) % self.k] = x   # STEP: the slot after the last item
        self.count += 1
        return True

    def deQueue(self):
        if self.count == 0:
            return False                     # empty
        self.head = (self.head + 1) % self.k # STEP: move the label, not the data
        self.count -= 1
        return True

    def Front(self):
        return self.buf[self.head] if self.count else -1

    def Rear(self):
        return self.buf[(self.head + self.count - 1) % self.k] if self.count else -1


cq = MyCircularQueue(3)
print([cq.enQueue(x) for x in [1, 2, 3, 4]], cq.Rear())    # [True, True, True, False] 3
print(cq.deQueue(), cq.enQueue(4), cq.Front(), cq.Rear())  # True True 2 4

**Try it**
- On a fresh `MyCircularQueue(3)`: enQueue 1, 2, 3, deQueue twice, enQueue 4 and 5, then print `cq.buf, cq.head`: `[4, 5, 3]` and 2. The new items wrapped into slots 0 and 1, and the front is still 3.
- Remove `% self.k` from `Rear` and call it in that state: `IndexError`, because `head + count - 1` is 4. Python forgives index -1, but not 4.
- Change `self.count == self.k` in `enQueue` to `self.count > self.k`: a 4th `enQueue` on a size-3 queue now succeeds and overwrites the oldest item (check `Front()`).

**Round robin** (649): each queue holds one party's senators in the order they act next, so the two fronts are the next senator of each side. The earlier one acts: it bans the other (who is never put back) and rejoins its own line one full round later, as `index + n`. Adding `n` keeps both lines sorted by real turn time.

In [ ]:
def predict_party_victory(senate):
    n = len(senate)
    r = deque(i for i, c in enumerate(senate) if c == "R")   # STATE + INIT: turn times, in order
    d = deque(i for i, c in enumerate(senate) if c == "D")
    while r and d:
        a, b = r.popleft(), d.popleft()      # the next senator of each party
        if a < b:
            r.append(a + n)                  # STEP: R acts first, bans b, comes back next round
        else:
            d.append(b + n)                  # STEP: D acts first, bans a, comes back next round
    return "Radiant" if r else "Dire"        # RETURN: the party that still has senators


print(predict_party_victory("RD"), predict_party_victory("RDD"), predict_party_victory("DDRRR"))   # Radiant Dire Dire

**Try it**
- Change `a + n` and `b + n` to plain `a` and `b`, then run `"RDD"`: `Radiant` instead of `Dire`. The R senator gets a second turn before D at index 2 has had its first.
- Print `list(r), list(d)` at the top of the loop for `"DDRRR"` and follow each ban.
- Predict `"RRDDD"` before running: `Radiant`, because both R senators act before any D and ban the first two.
- Count the loop rounds for a few strings: never more than `n - 1`, because each round removes one senator for good.

### Say it in the interview

Warm-up (Valid Parentheses): "a closer must match the most recent unfinished opener, so I push openers, compare each closer with the top, and the string is valid iff the stack ends empty: O(n)."

The medium version, Decode String:

> "The brute force expands an innermost `k[...]` and rescans, again and again. But a `]` always closes the most recent open `[`, which is last-in-first-out, so one pass with a stack works. At `[` I park the text built so far and the count, and start fresh. At `]` I pop and glue prefix + inner × count. The invariant: `cur` is the innermost open group's text so far, and the stack holds the outer prefixes, innermost on top. Time is proportional to the output; the stack holds one frame per open bracket."

Point at the two pushes and pops while you say the invariant. Likely follow-ups and your answers:

- *Counts with several digits* → `k = k * 10 + int(ch)`, and reset `k` after `[`.
- *Text after a group*, `"2[a]b"` → a letter always extends `cur`, so it just works: `aab`.
- *Recursion instead of a stack* → each `[` calls the decoder for the inner part; the call stack holds the same frames.
- *A stray `]` with nothing open* → check `if not stack` before popping and report invalid input.
- *A calculator instead* → `*` and `/` happen at once, on the top term; `+` and `-` wait for the final sum; `int(a / b)` because Python's `//` floors.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Asteroid Collision | `stack/asteroid_collision.py` · `practice/simple/basics/stacks/05_asteroid_collision.py` | survivors on a stack; a left-mover fights right-moving tops while it keeps winning |
| Basic Calculator | `stack/basic_calculator.py` | only + and −: a parenthesis just flips the signs inside it; keep a stack of context signs |
| Basic Calculator II | `stack/basic_calculator_ii.py` · `practice/simple/basics/stacks/04_basic_calculator_ii.py` | apply the previous operator when the next one arrives; push signed terms, `*` `/` edit the top, then sum |
| Decode String | `stack/decode_string.py` · `practice/simple/basics/stacks/03_decode_string.py` | on `[` park (text, count) and start fresh; on `]` pop and glue prefix + inner × count |
| Design Circular Deque | `queues/design_circular_deque.py` | ring buffer at both ends: insertFront steps head back `(head − 1) % k` before writing |
| Design Circular Queue | `queues/design_circular_queue.py` | fixed array + head + count; tail = (head + count) % k; count tells empty from full |
| Dota2 Senate | `queues/dota2_senate.py` | one queue of turn indices per party; the earlier front bans the other and rejoins as i + n |
| Evaluate Reverse Polish Notation | `stack/evaluate_reverse_polish_notation.py` · `practice/simple/15_evaluate_reverse_polish_notation.py` | numbers wait on a stack; an operator pops the right operand, then the left, and pushes the result |
| Implement Queue using Stacks | `queues/implement_queue_using_stacks.py` · `practice/simple/basics/stacks/01_array_stack_and_queue_via_two_stacks.py` | inbox + outbox; pour only when the outbox is empty, so each item moves once: amortized O(1) |
| Implement Stack using Queues | `queues/implement_stack_using_queues.py` | after each push, rotate the queue len − 1 times so the newest item sits at the front |
| Longest Valid Parentheses | `stack/longest_valid_parentheses.py` | indices on a −1 barrier; after a match, run = i − top; a stray `)` becomes the barrier |
| Min Stack | `stack/min_stack.py` · `practice/simple/14_min_stack.py` | store (value, min so far) in every entry; popping reveals the older minimum for free |
| Minimum Add to Make Parentheses Valid | `stack/minimum_add_to_make_parentheses_valid.py` | one bracket type: the stack is a counter; a `)` at zero costs one insert, leftover opens one each |
| Number of Recent Calls | `queues/number_of_recent_calls.py` | t only grows, so expired pings sit at the front: popleft while q[0] < t − 3000 |
| Parsing a Boolean Expression | `stack/parsing_a_boolean_expression.py` | push everything; on `)` pop letters into a set down to `(`, then apply the operator below it |
| Valid Parentheses | `stack/valid_parentheses.py` · `practice/simple/13_valid_parentheses.py` | push openers; a closer must match the top; valid iff the stack ends empty |

### Self-check

1. Why can't Valid Parentheses be solved with one counter per bracket type?
<details><summary>Answer</summary>Counters forget the order. <code>"([)]"</code> has one of each and balanced counts, but its <code>)</code> arrives while <code>[</code> is the innermost open bracket. A counter is enough only with a single bracket type, because then every stack entry is the same and only the stack's height matters (921).</details>

2. In Basic Calculator II, why is an operator applied when the *next* operator arrives, not when it is read?
<details><summary>Answer</summary>When you read <code>*</code>, its right operand hasn't been read yet. The number is complete only when the next operator (or the sentinel at the end) shows up, so the pending operator waits in <code>op</code> and is applied then.</details>

3. Queue from two stacks: one `pop` can move n items. Why is it still O(1)?
<details><summary>Answer</summary>Amortized: every item is pushed onto the inbox once, moved to the outbox once and popped once, a constant number of stack operations per item. An unlucky single pop does many moves at once, but n operations never do more than a constant times n moves in total.</details>

4. The ring buffer keeps `head` and `count`. What goes wrong with only `head` and `tail`?
<details><summary>Answer</summary><code>head == tail</code> happens both when the queue is empty and when it is full, so you can't tell them apart. Keeping <code>count</code> (or always leaving one slot unused) removes the ambiguity, and the tail is just <code>(head + count) % k</code>.</details>

<a id="s08"></a>

## Monotonic Stack

> Keep a stack of the items that are still **waiting** for their answer. Each new item looks at the top: if it beats that item, the waiting item has just found its answer (pop it and record), and the newcomer looks at the next one down. Then the newcomer waits too. Beaten items leave, so the waiting values are always in sorted order: that is the "monotonic".

**Reach for it when** the problem asks, for every element, about the **nearest** element to its left or right that is **bigger or smaller** (the next warmer day, the next greater element, the previous smaller one, how far a bar can stretch, a stock's span), about the minimum or maximum of **every subarray**, or about deleting digits or letters to get the **smallest or largest sequence** while keeping the order.

**In this repo:** `stack/` (6 of its 16 problems: daily temperatures, next greater II, the two rectangle problems, car fleet, create maximum number) · bank: `practice/simple/16_daily_temperatures.py`, `practice/simple/17_largest_rectangle_in_histogram.py` · basics in `practice/simple/basics/monotonic_stacks/`: `01_next_greater_element.py`, `02_previous_smaller_element.py`, `03_online_stock_span.py`, `04_sum_of_subarray_minimums.py`, `05_remove_k_digits.py`

### The picture

```text
temps   73  74  75  71  69  72  76  73       question: how many days until a warmer day?
day      0   1   2   3   4   5   6   7

the days still waiting, just before day 5:   day 5 (72) arrives and looks at the top:

    #                                          69 < 72: pop day 4, it waited 5 - 4 = 1 day
    #   #                                      71 < 72: pop day 3, it waited 5 - 3 = 2 days
    #   #   #                                  75 > 72: stop, day 2 keeps waiting
   75  71  69                                  push day 5: the waiting temps are now 75 72
   d2  d3  d4
```

The waiting days always form a staircase that never steps up. A newcomer knocks down every step shorter than itself (each knocked-down step has found its warmer day) and becomes the new last step.

**Why it is fast:** the brute force scans forward from every day: O(n²). On a falling stretch like 75 71 69, the scan from 75 walks over 71 and 69, which are *themselves still waiting* for a warmer day, so neither can be 75's answer. The stack flips the question: instead of each day searching forward for its answer, each new day announces itself as the answer to every waiting day it beats. Every index is pushed once and popped at most once, so the whole pass is O(n), even with a `while` inside the `for`.

**Why it is correct:** when day `i` pops day `j`, `j` was still waiting, so no day between them was warmer than `j` (it would have popped `j` first). So `i` is the *first* warmer day after `j`, which is exactly the answer. A day that is never popped never met a warmer day.

### From idea to code

**The idea in one sentence:** *for each new item: while it beats the item on top of the stack, that item is resolved (pop it, record its answer); then push the new item, which now waits for its own answer.*

| Decision | Monotonic-stack answer |
|---|---|
| **State / Definition** | a stack of **indices** of the items still waiting for their answer: `stack` = days whose warmer day hasn't come yet, temperatures never rising from bottom to top. Indices, not values: you need the position to write `ans[j]` and the distance `i - j` |
| **Invariant** | every index left of `i` that is *not* on the stack already has its answer; the values on the stack are sorted; and every day between a waiting day `j` and today was no warmer than `j`, or it would have popped `j` |
| **Step** | `stack.append(i)`: the newcomer waits for its own answer |
| **Fix** | before the step, `while stack and temps[stack[-1]] < t:` pop. The newcomer resolves (for *previous* questions: discards) every waiting item it beats |
| **Record** | *next* questions: at the pop (the popped index is answered by `i`). *Previous* questions: after the fix, before the step (the survivor on top answers `i`) |
| **Init** | `ans = [0] * n` or `[-1] * n` (the answer for "never"), `stack = []`; sometimes a sentinel item at the end to flush the stack |
| **Return** | `ans` (indices never popped keep the default), or the best value recorded at the pops |

**Which comparison?** Derive it from the question instead of memorising it. *Next* questions: pop while the newcomer *is* the top's answer, using the problem's own comparison ("strictly greater" → pop while `top < x`; "greater or equal" → pop while `top <= x`). *Previous* questions: pop while the top *can't be* the newcomer's answer (the negation: "strictly smaller" → pop while `top >= x`), then read the top. Each pop gives two facts at once: the popped item's next answer is the newcomer, and after the loop the top is the newcomer's previous answer, with the opposite strictness. Largest Rectangle uses both. The four common cases, as a check:

| For every item, find the nearest ... | Pop while the top is ... `x` | Values on the stack, bottom → top |
|---|---|---|
| greater to the **right** (next greater) | `< x` | never rising |
| smaller to the **right** (next smaller) | `> x` | never falling |
| greater to the **left** (previous greater) | `<= x`, then read the top | strictly falling |
| smaller to the **left** (previous smaller) | `>= x`, then read the top | strictly rising |

The same idea, sentence by sentence:

| In words | In code |
|---|---|
| "the items still waiting for an answer" | `stack = []` (indices) |
| "the newcomer beats the most recent waiting item" | `while stack and temps[stack[-1]] < t:` |
| "that item is resolved, and I am its answer" | `j = stack.pop()`, then `ans[j] = i - j` (or `ans[j] = t` for the value) |
| "now I wait too" | `stack.append(i)` |
| "my nearest smaller item on the left" | after popping everything `>= x`: `nums[stack[-1]] if stack else -1` |
| "nobody ever beat it" | the default left in `ans` |
| "resolve whatever still waits at the end" | a sentinel item: `heights + [0]` |

Two templates: *next* answers are written at the pop, *previous* answers after the loop. FIX runs before STEP: the newcomer must resolve or discard the waiting items before it waits itself. Pushed first, it would sit on top and block every comparison (`[73, 74]` → `[0, 0]`).

In [ ]:
def daily_temperatures(temps):
    ans = [0] * len(temps)                   # INIT: 0 = "no warmer day ever comes"
    stack = []                               # STATE: days still waiting; temps never rise bottom -> top
    for i, t in enumerate(temps):
        while stack and temps[stack[-1]] < t:    # FIX: today beats the latest waiting day
            j = stack.pop()
            ans[j] = i - j                   # RECORD: j is resolved; i is its FIRST warmer day
        stack.append(i)                      # STEP: today waits for its own warmer day
    return ans                               # RETURN: never resolved -> stays 0


def previous_smaller(nums):
    ans, stack = [], []                      # STATE + INIT: indices; values strictly rise bottom -> top
    for i, x in enumerate(nums):
        while stack and nums[stack[-1]] >= x:    # FIX: not smaller than x, and x is nearer: useless now
            stack.pop()
        ans.append(nums[stack[-1]] if stack else -1)   # RECORD: the survivor on top is the answer
        stack.append(i)                      # STEP: x may answer later items
    return ans                               # RETURN


print(daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]))   # [1, 1, 4, 2, 1, 1, 0, 0]
print(previous_smaller([3, 1, 4, 1, 5]))                      # [-1, -1, 1, -1, 1]

**Try it**
- Use the rule to turn `previous_smaller` into *previous greater* (pop while `nums[stack[-1]] <= x`) and predict `[3, 1, 4, 1, 5]` before running: `[-1, 3, -1, 4, -1]`.
- Push the temperature instead of the index (`stack.append(t)`) and run the first example: `IndexError`, because the stack's entries are used as positions.
- Print `stack` right before the `return` for the first example: `[6, 7]`, the two days that never got a warmer day.
- In `previous_smaller`, change `>=` to `>` and run `previous_smaller([2, 2])`: `[-1, 2]` instead of `[-1, -1]`. An equal value is not smaller.

### Watch it work

The richest use of the loop is Largest Rectangle in Histogram (84), where one pop answers *both* questions. Bars pop when a bar that is not taller arrives (`>=`), so the newcomer is the popped bar's right wall, and the bar under it on the stack is its left wall (its previous smaller bar, or -1, an imaginary wall before the array, if there is none). The rectangle with the popped bar's height spans the bars strictly between the walls. A final bar of height 0 pops whatever still waits.

In [ ]:
def trace_rectangle(heights):
    bars, stack, best = heights + [0], [], 0
    for i, h in enumerate(bars):
        while stack and bars[stack[-1]] >= h:
            j = stack.pop()
            left = stack[-1] if stack else -1
            area = bars[j] * (i - left - 1)
            best = max(best, area)
            print(f"  bar {i} (h={h}) pops bar {j} (h={bars[j]}): walls {left} and {i}, width {i - left - 1}, area {area}")
        stack.append(i)
        print(f"i={i} h={h}  stack heights: {[bars[k] for k in stack]}")
    return best


print(trace_rectangle([2, 1, 5, 6, 2, 3]))   # 10

**Try it**
- Find the line that gives 10: bar 4 pops bar 2 (height 5), walls 1 and 4, width 2.
- Run `trace_rectangle([3, 3, 3])`: equal bars pop each other with widths 1 and 2 (under-measured), and only the last one gets width 3 (area 9). Harmless, which is why `>` and `>=` give the same answer.
- Run `trace_rectangle([1, 2, 3])`: nothing pops until the height-0 bar arrives, then all three pop at `i = 3`, and the best is 4.

### Where it goes wrong

1. **Storing values instead of indices.** You lose the position, so there is no way to write `ans[j]` or compute `i - j`. Push `i`; read the value as `nums[stack[-1]]`.
2. **`<` vs `<=` when popping.** "Strictly warmer" pops on `<`. Popping on `<=` lets an equal temperature answer: `[70, 70, 71]` would give `[1, 1, 0]` instead of `[2, 1, 0]`.
3. **Recording in the wrong place.** *Next* answers are written for the **popped** index, at the pop. *Previous* answers are written for the **current** index, after the loop. Mixing them up gives answers to the wrong items.
4. **Forgetting the leftovers.** Items still on the stack at the end were never resolved. They need the default (`-1`, `0`), or, in the histogram, a final bar of height 0 so they get measured: `[1, 2, 3]` must give 4, but without the flush nothing is ever popped and you get 0.
5. **Width off by one.** When bar `j` pops at `i` and `left` is the index under it, the bars strictly between the walls number `i - left - 1`, not `i - left`.
6. **Counting ties twice.** In Sum of Subarray Minimums, `[2, 2]` has two equal minimums, and the subarray `[2, 2]` must be credited to exactly one of them: one wall strict, the other not. A single stack pass does this by itself; computing the left and right walls in two separate passes with the same comparison gives 8 or 4 instead of 6.
7. **Remove K Digits leftovers.** If the digits never fall (`"12345"`, k = 2), the loop pops nothing, so cut the last k digits at the end. Then strip leading zeros, and return `"0"` if nothing is left (`"10"`, k = 2).

### Edge cases to say out loud

Empty input · one item · all equal (nothing is strictly greater) · strictly rising (everything resolves at once) · strictly falling (nothing resolves; everything is left at the end) · ties. And one habit worth more than any list: check the fast version against a brute force on many small random inputs.

In [ ]:
assert daily_temperatures([]) == []
assert daily_temperatures([50]) == [0]
assert daily_temperatures([70, 70, 70]) == [0, 0, 0]        # equal is not warmer
assert daily_temperatures([30, 40, 50]) == [1, 1, 0]        # each one resolved at once
assert daily_temperatures([50, 40, 30]) == [0, 0, 0]        # nobody is ever resolved
assert previous_smaller([]) == []
assert previous_smaller([2, 2]) == [-1, -1]                 # equal is not smaller
assert previous_smaller([1, 2, 3]) == [-1, 1, 2]


def brute_daily(temps):                      # O(n²) reference: scan forward from every day
    return [next((j - i for j in range(i + 1, len(temps)) if temps[j] > temps[i]), 0)
            for i in range(len(temps))]


random.seed(0)
for _ in range(300):                         # small random inputs, full of ties
    T = [random.randint(30, 35) for _ in range(random.randint(0, 9))]
    assert daily_temperatures(T) == brute_daily(T), T
print("edge cases pass")

**Try it**
- Predict `previous_smaller([5, 4, 3])` and `previous_smaller([1, 3, 2])` before running: `[-1, -1, -1]` and `[-1, 1, 1]`.
- Break `daily_temperatures` on purpose (pop on `<=`) and rerun this cell: the `[70, 70, 70]` assert fails first. Delete that assert and rerun: the random check fails too, and its message shows the input it failed on.
- Add `assert daily_temperatures([50] * 100_000) == [0] * 100_000`: it runs instantly. Every index is pushed once and never popped.

### Variations

| Variation | What changes from the template | Problems |
|---|---|---|
| **Next greater value** | record `ans[j] = x` instead of the distance | 496 |
| **Next greater in a linked list** | copy the values into a list, then run the template | 1019 |
| **Circular array** | walk `2n` steps with `i % n`; push only during the first lap | 503 |
| **Previous smaller / greater** | pop the useless items, then read the top for the current item | 84, 901 |
| **Span, online** | store `(price, span)`; a popped day's span is absorbed into today's | 901 |
| **Both walls at once** | rising stack: when `j` pops at `i`, `i` is its right wall and the new top its left wall | 84, 85; Trapping Rain Water (42) has a stack version too, but is taught in [Two Pointers](03_Two_Pointers_Sliding_Window_Strings.ipynb#s05) |
| **Both walls + prefix sums** | each minimum's walls bound the widest subarray it rules; its sum comes from prefix sums | 1856 |
| **Contribution counting** | `nums[j]` is the minimum of `(j - left) * (i - j)` subarrays | 907; 2104 is 907 twice (maxima minus minima) |
| **Greedy smallest / largest sequence** | pop while the top is worse than `x` *and* you can still afford to drop it | 402, 1081, 321 |
| **Scan from the right** | 132 Pattern: walk right to left; the last value popped is the best "2" | 456 |
| **Candidates, then a second scan** | keep a falling stack of left candidates, then match them from the right | 962 |
| **Monotonic deque** | this stack plus expiry from the front: the window maximum | 239, in [Sliding Window](03_Two_Pointers_Sliding_Window_Strings.ipynb#s06) |
| **Sort first, then stack** | sort by position; a car whose solo arrival time is later than the fleet ahead's starts a new fleet | 853 |

**Circular array** (503): after the last element comes the first. Walk the array twice: the second lap lets the elements at the start answer the items still waiting at the end. Only the first lap pushes, so each index waits once.

In [ ]:
def next_greater_circular(nums):
    n = len(nums)
    ans, stack = [-1] * n, []                # STATE + INIT: indices still waiting; -1 = "never"
    for i in range(2 * n):                   # two laps around the circle
        x = nums[i % n]
        while stack and nums[stack[-1]] < x: # FIX + RECORD: x answers every smaller waiting item
            ans[stack.pop()] = x
        if i < n:
            stack.append(i)                  # STEP: only the first lap creates waiting items
    return ans                               # RETURN


print(next_greater_circular([1, 2, 1]), next_greater_circular([1, 2, 3, 4, 3]))   # [2, -1, 2] [2, 3, 4, -1, 4]

**Try it**
- Delete `if i < n:` so the second lap pushes too: `IndexError`, since `i` (not `i % n`) is then used as a position past the end of `nums`.
- Walk one lap only (`range(n)`) and run `[1, 2, 1]`: `[2, -1, -1]`. The last 1 never sees the 2 at the front.
- Predict `[5, 4, 3, 2, 1]` and `[3, 3, 3]` before running: `[-1, 5, 5, 5, 5]` and `[-1, -1, -1]`.

**Largest rectangle** (84, 85): the loop traced in Watch it work. Bar `j`'s best rectangle has its height and stretches until the first shorter bar on each side; the pop delivers both walls. Maximal Rectangle turns each row of the matrix into a histogram of the 1s standing on that row, then reuses the same function.

In [ ]:
def largest_rectangle(heights):
    bars = heights + [0]                     # a height-0 bar resolves everything at the end
    stack, best = [], 0                      # STATE + INIT: indices; heights rise bottom -> top
    for i, h in enumerate(bars):
        while stack and bars[stack[-1]] >= h:    # FIX: the top can't stretch past i
            height = bars[stack.pop()]
            left = stack[-1] if stack else -1    # its left wall: the nearest shorter bar
            best = max(best, height * (i - left - 1))   # RECORD: bars strictly between the walls
        stack.append(i)                      # STEP
    return best                              # RETURN


def maximal_rectangle(matrix):
    best, heights = 0, [0] * (len(matrix[0]) if matrix else 0)
    for row in matrix:
        for c, cell in enumerate(row):       # the 1s standing on this row, per column
            heights[c] = heights[c] + 1 if cell == "1" else 0
        best = max(best, largest_rectangle(heights))
    return best


grid = [["1", "0", "1", "0", "0"], ["1", "0", "1", "1", "1"],
        ["1", "1", "1", "1", "1"], ["1", "0", "0", "1", "0"]]
print(largest_rectangle([2, 1, 5, 6, 2, 3]), largest_rectangle([2, 4]), maximal_rectangle(grid))   # 10 4 6

**Try it**
- Drop the sentinel (`bars = heights`) and run `largest_rectangle([1, 2, 3])`: 0 instead of 4. The heights only rose, so no bar was ever popped and measured.
- Change the width to `i - left` and run `[2, 1, 5, 6, 2, 3]` again: 15 instead of 10. Every rectangle now counts one bar too many (height 5 gets width 3).
- Print `heights` after each row in `maximal_rectangle(grid)`: `[1, 0, 1, 0, 0]`, `[2, 0, 2, 1, 1]`, `[3, 1, 3, 2, 2]`, `[4, 0, 0, 3, 0]`. A `"0"` resets its column to the ground.

**Sum of subarray minimums** (907): turn the sum around. Instead of finding the minimum of every subarray, ask for every element *how many subarrays it is the minimum of*. With its walls at `left` and `i` (the nearest smaller items), a subarray has `nums[j]` as its minimum when it starts in `left+1 .. j` and ends in `j .. i-1`: that is `(j - left) * (i - j)` subarrays. The histogram's pop hands you both walls.

In [ ]:
def sum_subarray_mins(nums):
    vals = nums + [-math.inf]                # sentinel: smaller than everything, pops all
    stack, total = [], 0                     # STATE + INIT: indices; values rise bottom -> top
    for i, x in enumerate(vals):
        while stack and vals[stack[-1]] >= x:    # FIX
            j = stack.pop()                  # i = right wall: the first value <= vals[j]
            left = stack[-1] if stack else -1    # left wall: the last value < vals[j]
            total += vals[j] * (j - left) * (i - j)   # RECORD: vals[j] is the min of that many subarrays
        stack.append(i)                      # STEP
    return total % (10**9 + 7)               # RETURN


print(sum_subarray_mins([3, 1, 2, 4]), sum_subarray_mins([11, 81, 94, 43, 3]), sum_subarray_mins([2, 2]))   # 17 444 6

**Try it**
- Check it against the brute force: with `nums = [3, 1, 2, 4]`, `sum(min(nums[i:j]) for i in range(4) for j in range(i + 1, 5))` is also 17.
- Print `(vals[j], j - left, i - j)` at each pop for `[3, 1, 2, 4]`: the 1 is the minimum of 2 × 3 = 6 subarrays.
- Change `>=` to `>`: still 17, and still 6 for `[2, 2]`. One pass always makes one wall strict and the other not, so a tie is counted once either way.

**Smallest sequence by deleting** (402, 1081, 321): to make the smallest number, the leftmost digits matter most. A digit sitting right before a smaller one should go, because deleting it slides the smaller digit into a more important place. So the digits you keep should never fall: pop while deletions are left. Smallest Subsequence of Distinct Characters (1081, same as 316) swaps the budget for a different test: drop a letter only if it appears again later. Create Maximum Number (321) uses the mirror image to keep the *largest* t digits.

In [ ]:
def remove_k_digits(num, k):
    stack = []                               # STATE + INIT: kept digits, never falling
    for d in num:
        while k and stack and stack[-1] > d: # FIX: a bigger digit before a smaller one: drop it
            stack.pop()
            k -= 1
        stack.append(d)                      # STEP
    stack = stack[:len(stack) - k]           # deletions left over come off the tail
    return "".join(stack).lstrip("0") or "0" # RETURN


def smallest_subsequence(s):                 # every distinct letter exactly once
    last = {c: i for i, c in enumerate(s)}   # the last position of each letter
    stack, used = [], set()
    for i, c in enumerate(s):
        if c in used:
            continue
        while stack and stack[-1] > c and last[stack[-1]] > i:   # FIX: it comes back later: drop it
            used.remove(stack.pop())
        stack.append(c)                      # STEP
        used.add(c)
    return "".join(stack)


def max_subsequence(nums, t):                # 321's building block: the largest t digits, in order
    drop, stack = len(nums) - t, []
    for x in nums:
        while drop and stack and stack[-1] < x:   # FIX: the mirror image, drop smaller digits
            stack.pop()
            drop -= 1
        stack.append(x)                      # STEP
    return stack[:t]


print(remove_k_digits("1432219", 3), remove_k_digits("10200", 1), remove_k_digits("10", 2))   # 1219 200 0
print(smallest_subsequence("bcabc"), smallest_subsequence("cbacdcbc"))                      # abc acdb
print(max_subsequence([9, 1, 2, 5, 8, 3], 3), max_subsequence([3, 4, 6, 5], 2))             # [9, 8, 3] [6, 5]

**Try it**
- Delete the line that trims leftover deletions and run `remove_k_digits("12345", 2)`: `12345` instead of `123`.
- Drop `or "0"` and run `remove_k_digits("10", 2)`: an empty string instead of `0`.
- In `smallest_subsequence`, delete `and last[stack[-1]] > i` and run `"cbacdcbc"`: `abc`. The `d` was dropped although it never comes back.
- Predict `max_subsequence([6, 0, 4], 2)` before running: `[6, 4]`.

**Span and fleets** (901, 853): a stock's span asks for the *previous greater* price, online: how many days in a row, ending today, had a price ≤ today's? A day that today beats lies inside today's span, and so does every day it had already absorbed, so store `(price, span)` pairs and add the spans you pop. Car Fleet sorts by position and walks from the car closest to the target, comparing *solo arrival times*, not speeds. A car whose time is later than the fleet ahead's can never catch it and leads a new fleet; one that would arrive at the same time or sooner catches up and merges. The times on that stack only rise, so it never pops: it is really just a running maximum.

In [ ]:
class StockSpanner:                          # 901
    def __init__(self):
        self.stack = []                      # STATE + INIT: (price, span); prices strictly fall bottom -> top

    def next(self, price):
        span = 1                             # today itself
        while self.stack and self.stack[-1][0] <= price:   # FIX: absorb every day today beats
            span += self.stack.pop()[1]      # ... and every day that day had covered
        self.stack.append((price, span))     # STEP
        return span                          # RECORD + RETURN


def car_fleet(target, position, speed):      # 853
    times = []                               # STATE + INIT: arrival times of the fleets, front fleet first
    for pos, spd in sorted(zip(position, speed), reverse=True):   # closest to the target first
        t = (target - pos) / spd             # this car's solo arrival time
        if not times or t > times[-1]:       # arrives later than the fleet ahead: a new fleet
            times.append(t)                  # STEP (otherwise it catches up and merges)
    return len(times)                        # RETURN


spanner = StockSpanner()
print([spanner.next(p) for p in [100, 80, 60, 70, 60, 75, 85]])   # [1, 1, 1, 2, 1, 4, 6]
print(car_fleet(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]))           # 3

**Try it**
- Change `<=` to `<` in `next` and replay `[60, 60, 60]` on a fresh spanner: `[1, 1, 1]` instead of `[1, 2, 3]`.
- Count each popped day as 1 (`span += 1`) instead of adding its span, and replay the main prices: `[1, 1, 1, 2, 1, 3, 3]`. The 75 loses the day that the 70 had absorbed, and the 85 gets 3 instead of 6.
- In `car_fleet`, drop `reverse=True` and rerun: 1 instead of 3. Each car now compares itself with a car *behind* it.
- Run `car_fleet(10, [0, 5], [2, 1])`: 1, because both cars reach the target at t = 5 and arrive together. With `t >= times[-1]` it would say 2.

### Say it in the interview

> "The brute force scans right from every item for the first bigger one: O(n²). Those scans keep walking over items that are themselves still waiting. So I'll flip it: a stack holds the indices still waiting, whose values never rise; each new item pops and answers every smaller item on top, then waits itself. When `i` pops `j`, `j` was still waiting, so nothing between them beat it: `i` is `j`'s first warmer day. Every index is pushed once and popped once, so it's O(n) time and O(n) space."

Then point at the pop line and say *who* is resolved there ("this index just met its next warmer day"), and say what is left on the stack at the end and which answer those items get. Likely follow-ups and your answers:

- *Values instead of distances* (496) → record `ans[j] = t`.
- *A circular array* (503) → walk two laps with `i % n`; push only during the first.
- *The previous item instead of the next* → record after the loop, before pushing.
- *A stream of prices* (901) → the same stack, kept between calls; each call is amortized O(1).
- *O(1) extra space for 739* → scan right to left; to find day `i`'s answer, start at `j = i + 1` and jump `j += ans[j]` over days that are not warmer.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Car Fleet | `stack/car_fleet.py` | sort by position, closest first; compare solo arrival times: later than the fleet ahead → new fleet, same or sooner → merges |
| Create Maximum Number | `stack/create_maximum_number.py` | try every split; best subsequence per array by monotonic stack; merge by comparing remaining tails, not heads |
| Daily Temperatures | `stack/daily_temperatures.py` · `practice/simple/16_daily_temperatures.py` | stack of days still waiting (temperatures never rising); a warmer day pops them and records i − j |
| Largest Rectangle in Histogram | `stack/largest_rectangle_in_histogram.py` · `practice/simple/17_largest_rectangle_in_histogram.py` | rising stack; a popped bar's walls are the current index and the new top; flush with a 0 bar |
| Maximal Rectangle | `stack/maximal_rectangle.py` | each row is a histogram of the 1s standing on it; run Largest Rectangle once per row |
| Next Greater Element | `practice/simple/basics/monotonic_stacks/01_next_greater_element.py` | waiting indices; a bigger value pops and answers them; leftovers keep −1 |
| Next Greater Element II | `stack/next_greater_element_ii.py` | circular: walk 2n steps with i % n and push only during the first lap |
| Online Stock Span | `practice/simple/basics/monotonic_stacks/03_online_stock_span.py` | (price, span) pairs; a price ≤ today's is absorbed with every day it covered |
| Previous Smaller Element | `practice/simple/basics/monotonic_stacks/02_previous_smaller_element.py` | pop everything ≥ x (x is nearer and no bigger), then the top is the answer |
| Remove K Digits | `practice/simple/basics/monotonic_stacks/05_remove_k_digits.py` | drop a digit bigger than the next while k lasts; trim the tail; strip zeros; "0" if empty |
| Sum of Subarray Minimums | `practice/simple/basics/monotonic_stacks/04_sum_of_subarray_minimums.py` | nums[j] is the min of (j − left)(i − j) subarrays; one wall strict, the other not |

### Self-check

1. Why does the stack hold indices rather than values?
<details><summary>Answer</summary>The answer is written for a specific position (<code>ans[j]</code>) and often needs a distance or a width (<code>i - j</code>, <code>i - left - 1</code>). A value loses its position, while an index can always give the value back as <code>nums[j]</code>.</details>

2. In Daily Temperatures, when exactly is a day resolved, and what happens to the days still on the stack at the end?
<details><summary>Answer</summary>A day is resolved at the moment it is popped: the current day is the first one warmer than it, so its answer is <code>i - j</code>. Days still waiting at the end never met a warmer day, so they keep the default 0.</details>

3. The loop has a `while` inside a `for`. Why is it O(n) and not O(n²)?
<details><summary>Answer</summary>Count pops instead of loops. Each index is pushed exactly once, so it can be popped at most once, and all the <code>while</code> loops together pop at most n times. The total work is O(n).</details>

4. In Largest Rectangle, why is the left wall of a popped bar simply the new top of the stack?
<details><summary>Answer</summary>It is <code>previous_smaller</code> in disguise. When bar j arrived, it popped every bar at least as tall, so the bar left under it is its previous shorter bar, and nothing under an entry changes while that entry waits.</details>

<a id="s10"></a>

## Linked Lists

> A linked list is a chain of boxes, each holding a value and **one** arrow to the next box. You can't index into it, walk it backwards or see its length; all you can do is put named fingers on boxes (`prev`, `cur`, `slow`, `fast`) and re-aim arrows. Every algorithm is careful pointer surgery: **hold on to whatever you are about to lose, then cut.**

**Reach for it when** the input is a `ListNode`; the problem says *in place* or *O(1) extra space*; it asks for the **middle**, the **k-th from the end**, a **cycle**, a **merge** or a **reversal**; or an array's values are indices into the same array (then the array *is* a linked list, and Floyd's cycle trick applies).

**In this repo:** `linked_list/` (12 problems) · bank: `practice/simple/21_reverse_linked_list.py`, `practice/simple/22_merge_two_sorted_lists.py`, `practice/simple/23_linked_list_cycle_ii.py`, `practice/simple/25_remove_nth_node_from_end.py` · basics: `practice/simple/basics/linked_lists/` (build/insert/delete, reverse iterative and recursive, middle, Floyd, merge, palindrome, k-group, add two numbers). LRU Cache (a dict plus a doubly linked list) is taught in the Design section.

### The picture

```text
A box holds a value and ONE arrow. A variable is a finger pointing at a box.

  head
   v
  [1]->[2]->[3]->None

Delete 2: put a finger on the box BEFORE it and re-aim that box's arrow past the victim.

  prev
   v
  [1]  [2]->[3]->None          prev.next = prev.next.next
   |         ^                 (nothing points at 2 any more: it is gone)
   +---------+

Insert 9 after 1: aim the new box first, then re-aim prev. The other order loses 3.

  [1]       [3]->None          node.next = prev.next      (9 -> 3)   first ...
   |         ^                 prev.next = node           (1 -> 9)   ... then this
   +->[9]----+

A dummy box in front gives even the head a box before it:

  [D]->[1]->[2]->None          deleting the head is just dummy.next = dummy.next.next,
                               and the answer is always dummy.next
```

The fingers can also *measure*, which is how a list answers questions an array answers with an index:

```text
middle: slow moves 1 box, fast moves 2; when fast can't take two more steps, slow is halfway

  [1]->[2]->[3]->[4]->[5]->None
             ^         ^
           slow      fast

n-th from the end (n = 2): open a gap of n + 1 links, then slide both until fast falls off

  [D]->[1]->[2]->[3]->[4]->[5]->None
                  ^               ^
                slow            fast          slow.next (4) is 2nd from the end: skip it

reverse: flip one arrow per step; the list is two chains that meet at the prev | cur split

  None<-[1]<-[2]    [3]->[4]->None
              ^      ^
            prev    cur
```

Why it is fast (and small): the brute force for almost every list problem is "copy the nodes into a Python list, use indices, rebuild". That is O(n) time but O(n) extra space, and the interviewer will ask you to drop it. Each list technique replaces one array superpower with a pointer trick: *index from the end* becomes a finger trailing by a fixed gap; *the middle* becomes a finger at half speed; *walk backwards* becomes "reverse that half in place"; *have I been here before?* becomes a fast finger lapping a slow one. Fingers only move forward, so each trick is O(n) time and O(1) space.

### From idea to code

**The idea in one sentence:** *put a finger on every box you must not lose, re-aim arrows in the order save → rewire → advance, and put a dummy box in front so the head is never a special case.*

| Decision | Linked-list answer |
|---|---|
| **State**: what must I remember? | a few fingers: `prev`/`cur` to rewire, `tail` to build a new chain, `slow`/`fast` to measure; often a `dummy` box in front of the head |
| **Definition**: what exactly does each variable mean? | say it in a comment: `prev` = head of the already-reversed part, `cur` = first box not processed yet, `tail` = last box of the answer so far |
| **Invariant**: what is true at the end of every step? | everything before `cur` is done and reachable from `prev` (or `dummy`); everything from `cur` on is untouched and reachable from `cur` |
| **Step**: how does one box change the state? | save `nxt = cur.next`, rewire `cur.next`, then advance the fingers: always in that order |
| **Record**: when is the answer updated? | building: hang a box on `tail`. Rewiring: nothing to record, the structure *is* the answer. Finding: the moment two fingers meet (`slow is fast`) |
| **Init**: starting values | `dummy = ListNode(0, head)`; `prev = None, cur = head` for a reversal; `slow = fast = head` (or `dummy`) for measuring |
| **Return**: what comes back? | `dummy.next` (never `head`: the head may have moved or been deleted), `prev` after a reversal, a box or `None` for a search |

The same idea, sentence by sentence:

| In words | In code |
|---|---|
| "a box in front of the head" | `dummy = ListNode(0, head)` |
| "hold on to the rest of the list" | `nxt = cur.next` |
| "flip this arrow backwards" | `cur.next = prev` |
| "step forward" | `prev, cur = cur, nxt` |
| "skip (delete) the next box" | `prev.next = prev.next.next` |
| "hang a box on the end of the answer" | `tail.next = node` then `tail = tail.next` |
| "can fast take two more steps?" | `while fast and fast.next:` |
| "the same box" (not the same value) | `a is b` |
| "the answer starts after the dummy" | `return dummy.next` |

First the box itself, and two helpers to turn a Python list into a linked list and back:

In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next


def build_list(values):                      # [1, 2, 3] -> 1 -> 2 -> 3, returns the head
    dummy = tail = ListNode()
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next


def to_list(head, limit=1000):               # 1 -> 2 -> 3 -> [1, 2, 3]
    out = []
    while head and len(out) < limit:         # the limit stops a walk around a cycle
        out.append(head.val)
        head = head.next
    return out


head = build_list([1, 2, 3])
print(head.val, head.next.val, head.next.next.next)   # 1 2 None
print(to_list(head))                                  # [1, 2, 3]

**Try it**
- `build_list([])` returns `None` (an empty list is just "no first box"), and `to_list(None)` is `[]`. Check both.
- Make a cycle by hand: `h = build_list([1, 2, 3]); h.next.next.next = h`, then `to_list(h, 7)` gives `[1, 2, 3, 1, 2, 3, 1]`. Without the limit this walk would never end: remember that when a bug creates a cycle.
- Rewrite `build_list` without the dummy. You will need an `if head is None:` branch for the first box: that branch is exactly what the dummy saves you.

Three templates cover most problems: rewire in place (reverse), build a new chain (merge), and fingers that measure (middle, n-th from the end). The tags are the seven decisions, and the **order** of the tagged lines is a decision too: `reverse_list` must save → flip → advance (flip first and the rest of the list is lost); `merge_lists` hangs a box on the answer (RECORD) before `tail` steps onto it (STEP); `remove_nth_from_end` records only after the slide, when `slow` is known to sit just before the victim.

In [ ]:
def reverse_list(head):
    prev, cur = None, head                   # STATE + INIT: prev = reversed part, cur = the rest
    while cur:
        nxt = cur.next                       # STEP 1: save the rest before cutting
        cur.next = prev                      # STEP 2: flip one arrow
        prev, cur = cur, nxt                 # STEP 3: both fingers move one box right
    return prev                              # RETURN: the old last box is the new head


def merge_lists(a, b):
    dummy = tail = ListNode()                # STATE + INIT: tail = last box of the answer
    while a and b:
        if a.val <= b.val:                   # the smaller front box goes next
            tail.next, a = a, a.next         # RECORD: hang it on the answer
        else:
            tail.next, b = b, b.next
        tail = tail.next                     # STEP: tail moves onto the box just hung
    tail.next = a or b                       # the leftover is already sorted: hang it whole
    return dummy.next                        # RETURN: skip the dummy


def middle_node(head):
    slow = fast = head                       # STATE + INIT: both fingers on the head
    while fast and fast.next:                # while fast can take two more steps
        slow, fast = slow.next, fast.next.next   # STEP: slow moves 1 box, fast moves 2
    return slow                              # RETURN: the middle (the 2nd middle if even)


def remove_nth_from_end(head, n):
    dummy = ListNode(0, head)                # INIT: a box before the head, so the head can go too
    slow = fast = dummy                      # STATE: two fingers that will stay n + 1 links apart
    for _ in range(n + 1):                   # INIT: open the gap
        fast = fast.next
    while fast:                              # until fast falls off the end
        slow, fast = slow.next, fast.next    # STEP: both move one box; the gap stays n + 1
    slow.next = slow.next.next               # RECORD: slow is just before the victim; skip it
    return dummy.next                        # RETURN: never `head` (it may be gone)


print(to_list(reverse_list(build_list([1, 2, 3, 4, 5]))))                   # [5, 4, 3, 2, 1]
print(to_list(merge_lists(build_list([1, 2, 4]), build_list([1, 3, 4]))))   # [1, 1, 2, 3, 4, 4]
print(middle_node(build_list([1, 2, 3, 4, 5])).val, middle_node(build_list([1, 2, 3, 4])).val)   # 3 3
print(to_list(remove_nth_from_end(build_list([1, 2, 3, 4, 5]), 2)))         # [1, 2, 3, 5]

**Try it**
- Swap the first two lines inside `reverse_list`'s loop (flip, *then* save) and reverse `[1, 2, 3, 4, 5]`: you get `[1]`. After the flip, `cur.next` is `None`, so `nxt` saved nothing and the rest of the list is lost.
- In `remove_nth_from_end`, change `range(n + 1)` to `range(n)`: `[1, 2, 3, 4, 5], 2` now gives `[1, 2, 3, 4]`. With a gap of n, `slow` stops *on* the 4 instead of before it, so the wrong box is skipped.
- Change `while fast and fast.next:` to `while fast.next:` and call `middle_node(build_list([1, 2, 3, 4]))`: `AttributeError`, because on an even length `fast` itself becomes `None`. Odd lengths still work, which is why this bug survives a quick test.
- Delete `tail.next = a or b` from `merge_lists`: the merge prints `[1, 1, 2, 3, 4]`. The second list's `4` was still waiting when the first list ran out, so it was never hung on the answer.

**Recursive reversal.** Interviewers sometimes ask for it. Trust the recursion: it reverses everything after `head` and returns the new head (the old last box). At that moment `head` still points at its old next box, which is now the *tail* of the reversed part, so hang `head` behind it:

```text
call on 1:     1 -> 2 -> 3 -> 4 -> None
the recursive call reverses 2 -> 3 -> 4 and returns 4:

               1 -> 2 <- 3 <- 4           1 still points at 2, and 2 is now the TAIL
                    |
                   None

head.next.next = head     1 <-> 2         2 points back at 1 (a 2-box loop, for a moment)
head.next = None          None <- 1 <- 2 <- 3 <- 4          return 4
```

In [ ]:
def reverse_recursive(head):
    if head is None or head.next is None:    # 0 or 1 box: already reversed
        return head
    new_head = reverse_recursive(head.next)  # trust it: everything after head is reversed
    head.next.next = head                    # my old next box is now the tail: hang me after it
    head.next = None                         # I am the new tail
    return new_head                          # the old last box, handed up unchanged


print(to_list(reverse_recursive(build_list([1, 2, 3, 4]))))   # [4, 3, 2, 1]

**Try it**
- Print `head.val, new_head.val` just before `return new_head`: you see `3 4`, `2 4`, `1 4`. The same box travels up through every level; only the arrows around `head` change.
- Delete `head.next = None` and rerun: the printed list becomes `[4, 3, 2, 1, 2, 1, 2, 1, ...]` and only stops at `to_list`'s limit. Boxes 1 and 2 now point at each other: a cycle.
- Try `reverse_recursive(build_list(range(10_000)))`: `RecursionError` (Python allows about 1000 nested calls by default). `reverse_list` handles the same list without trouble: prefer the loop unless recursion is asked for.

### Watch it work

The reversal keeps two chains: the reversed part (drawn with arrows pointing left, `prev` is its rightmost box) and the untouched rest (starting at `cur`). Each step moves one box from the right chain to the left one.

In [ ]:
def trace_reverse(values):
    prev, cur = None, build_list(values)

    def show(step):
        done = " <- ".join(["None"] + [str(v) for v in reversed(to_list(prev))])
        rest = " -> ".join([str(v) for v in to_list(cur)] + ["None"])
        print(f"step {step}:  {done:<26}|  {rest}")

    show(0)
    step = 0
    while cur:
        nxt = cur.next
        cur.next = prev
        prev, cur = cur, nxt
        step += 1
        show(step)


trace_reverse([1, 2, 3, 4])

**Try it**
- Run `trace_reverse([7])` and `trace_reverse([])`: one step for one box, and for the empty list only the start line (`prev` stays `None`, which is the right answer).
- On every line the two chains together hold all n boxes, with no box in both. That is the invariant: nothing is ever lost, it only changes sides.
- Change the loop to `while cur.next:` and run `trace_reverse([1, 2, 3])`: the last line still shows `3 -> None` on the right. The last box is never flipped, so you would return `2 -> 1` and lose the 3.

### Where it goes wrong

1. **Cutting before saving.** `cur.next = prev` before `nxt = cur.next` loses the rest of the list: `[1, 2, 3]` "reverses" to `[1]`. Fix: save, rewire, advance.
2. **Returning the old head.** After a reversal `head` is the *tail*: returning it gives `[1]` for `[1, 2, 3]`. Return `prev` (reversal) or `dummy.next` (anything built behind a dummy).
3. **Deleting or inserting at the head without a dummy.** `remove_nth_from_end([1], 1)` needs a box *before* the head. A dummy gives every box a predecessor; then return `dummy.next`.
4. **`fast.next.next` on `None`.** Guard with `while fast and fast.next:`. A loop that only checks `fast.next` crashes on `[1, 2, 3, 4]`.
5. **Gap off by one.** For "n-th from the end", open a gap of n + 1 links starting from the dummy so `slow` stops *before* the victim. A gap of n deletes the wrong box (`[1, 2, 3, 4, 5], 2` → `[1, 2, 3, 4]`).
6. **Values instead of boxes.** Two different boxes can hold the same value. Meeting points and intersections compare boxes: `a is b`, never `a.val == b.val`.
7. **Forgetting to cut.** When you split a list in two (reorder, palindrome, merge sort), set the first half's last arrow to `None`. Otherwise the halves still touch and a later rewire can build a cycle.
8. **Comparing before moving in Floyd.** `slow` and `fast` both start on the head, so `slow is fast` is true before the first step. Move first, then compare.
9. **Deep recursion.** Recursive reversal makes one call per box, and Python stops at about 1000 nested calls. Use the loop unless asked otherwise.

### Edge cases to say out loud

Empty list (`None`) · one box · two boxes (even length for fast/slow) · remove the head (n = length) · remove the last box (n = 1) · lists of different lengths · duplicate values (compare boxes, not values) · a cycle through the head, or a box pointing at itself.

In [ ]:
assert to_list(reverse_list(None)) == [] and to_list(reverse_list(build_list([7]))) == [7]
assert to_list(reverse_recursive(build_list([1, 2]))) == [2, 1]
assert to_list(merge_lists(None, build_list([0]))) == [0]
assert to_list(merge_lists(build_list([5]), build_list([1, 2, 3]))) == [1, 2, 3, 5]
assert middle_node(build_list([1])).val == 1
assert middle_node(build_list([1, 2])).val == 2                        # second middle
assert to_list(remove_nth_from_end(build_list([1]), 1)) == []          # the only box
assert to_list(remove_nth_from_end(build_list([1, 2]), 2)) == [2]      # the head
assert to_list(remove_nth_from_end(build_list([1, 2]), 1)) == [1]      # the last box
print("edge cases pass")

**Try it**
- Predict, then add: `assert middle_node(None) is None` (the `while` never runs, so `slow` is still `None`).
- What does `remove_nth_from_end(build_list([1, 2, 3]), 3)` return? Write the assert first: `[2, 3]`. When n equals the length you remove the head, the case the dummy exists for.
- Change `<=` to `<` in `merge_lists` and rerun: every assert still passes. On ties it only changes *which* of two equal boxes goes first (keeping `<=` keeps the merge stable).

### Variations

| Variation | What changes from the template | Problems |
|---|---|---|
| **Cycle: is there one, where does it start?** | fast/slow race; after they meet, restart one finger at the head and step both by 1 | 141, 142 |
| **Floyd on an array** | the "next" of index `i` is `nums[i]`; the duplicate value is the cycle entrance | 287 |
| **Fold in half** | middle → cut → reverse the back half → walk both halves together | 234, 143 |
| **Reverse a block** | reverse k boxes with `prev` starting at the box *after* the block, then hook the block in | 25 |
| **Two lists in lockstep** | one finger per list, step together (merge, add with a carry, line up two tails) | 21, 2, 160 |
| **Copy with extra arrows** | dict old box → new box, then wire `next` and `random` through it | 138 |
| **Recency order with O(1) moves** | dict + doubly linked list with two sentinels (Design section) | 146 |

**Cycles (141, 142).** Inside a loop, `fast` gains exactly one box per step on `slow`, so the gap between them shrinks to zero: they must meet. Without a loop, `fast` falls off the end. The surprising part is phase 2: why does a finger restarted at the head meet the other finger exactly at the entrance? Measure the rho shape:

```text
 head ------ a boxes ------> E ------ b boxes ------> M        E = entrance, M = where they met
                             ^                        |        c = boxes around the loop
                             +----- c - b boxes ------+

 when they meet:   slow walked a + b,   fast walked 2(a + b)
 fast's extra a + b steps were whole laps of the loop:     a + b = k * c
 so   a = k*c - b = (c - b) + (k - 1)*c   ->  walking a steps from M also ends at E
```

So one finger from the head and one from `M`, both stepping 1, arrive at `E` together. **Floyd on an array (287):** read index `i` as a box whose arrow points to index `nums[i]`. Values are in `[1, n]`, so no arrow ever enters index 0: it is the start of the tail. Two equal values are two arrows into the same box, and that box is the loop's entrance.

```text
 nums  = [1, 3, 4, 2, 2]
 index:   0  1  2  3  4

 0 -> 1 -> 3 -> 2 -> 4           boxes 3 and 4 both point at 2: two arrows enter 2,
                ^    |           so 2 is the duplicate AND the entrance of the loop
                +----+
```

In [ ]:
def build_cycle(values, pos):                # the last box links back to index pos (-1: no cycle)
    nodes = [ListNode(v) for v in values]
    for a, b in zip(nodes, nodes[1:]):
        a.next = b
    if pos >= 0:
        nodes[-1].next = nodes[pos]
    return nodes[0]


def detect_cycle(head):
    slow = fast = head
    while fast and fast.next:                # phase 1: race
        slow, fast = slow.next, fast.next.next
        if slow is fast:                     # met inside the loop (compare boxes, not values)
            break
    if fast is None or fast.next is None:    # the race ended because fast fell off: no cycle
        return None
    finger = head                            # phase 2: from the head and from the meeting point
    while finger is not slow:
        finger, slow = finger.next, slow.next
    return finger                            # the entrance


def find_duplicate(nums):                    # index i points to index nums[i]
    slow = fast = 0                          # 0 is never a target: it starts the tail
    while True:
        slow, fast = nums[slow], nums[nums[fast]]
        if slow == fast:                     # indices, so == is "same box" here
            break
    finger = 0
    while finger != slow:
        finger, slow = nums[finger], nums[slow]
    return finger                            # the box with two incoming arrows


print(detect_cycle(build_cycle([3, 2, 0, -4], 1)).val, detect_cycle(build_cycle([1, 2], -1)))   # 2 None
print(find_duplicate([1, 3, 4, 2, 2]), find_duplicate([3, 1, 3, 4, 2]))                          # 2 3

**Try it**
- Return `slow` right after phase 1 (skip phase 2) and rerun the first example: you get `-4`, the box where the race happened to end, not the entrance `2`.
- Check the formula on a bigger rho: in `build_cycle(list(range(10)), 3)` the tail has a = 3 boxes and the loop c = 7. Print `slow.val` after phase 1: `7`, so b = 4 and a + b = 7 is exactly one lap.
- Compare before moving: put the `if slow is fast: break` lines *above* the step line in phase 1. The cell now prints `3` and a `ListNode` instead of `2 None`: the fingers "meet" on the head before taking a single step.
- Predict `find_duplicate([2, 5, 9, 6, 9, 3, 8, 9, 7, 1])` before running (9: it appears three times, so three arrows enter box 9).

**Fold in half (234, 143).** Palindrome and Reorder both need "the back half, walked backwards". You can't walk backwards, but you can *make* the back half run backwards by reversing it in place. Recipe: find the middle, cut, reverse the back half, then walk both halves together.

```text
 1 -> 2 -> 3 -> 4 -> 5         slow stops on 3, the last box of the first half
 1 -> 2 -> 3     5 -> 4        cut after 3 and reverse the back half
 1 -> 5 -> 2 -> 4 -> 3         zip: one box from the left half, one from the right
```

In [ ]:
def reorder_list(head):                      # L0 -> Ln -> L1 -> Ln-1 -> ...  in place
    if head is None:
        return
    slow = fast = head
    while fast.next and fast.next.next:      # slow ends on the LAST box of the first half
        slow, fast = slow.next, fast.next.next
    second = slow.next
    slow.next = None                         # cut: the halves must not touch
    second = reverse_list(second)            # the back half, now running backwards
    first = head
    while second:                            # the first half is as long or one longer
        n1, n2 = first.next, second.next     # save both rests before rewiring
        first.next, second.next = second, n1
        first, second = n1, n2


def is_palindrome_list(head):
    second = reverse_list(middle_node(head)) # back half reversed (it includes the middle if odd)
    first = head
    while second:                            # the reversed half is never the longer one
        if first.val != second.val:
            return False
        first, second = first.next, second.next
    return True


h = build_list([1, 2, 3, 4, 5]); reorder_list(h); print(to_list(h))      # [1, 5, 2, 4, 3]
h = build_list([1, 2, 3, 4]); reorder_list(h); print(to_list(h))         # [1, 4, 2, 3]
print(is_palindrome_list(build_list([1, 2, 2, 1])), is_palindrome_list(build_list([1, 2])))   # True False

**Try it**
- Delete the cut `slow.next = None` and rerun: both reordered lists now run on until `to_list`'s limit. `to_list(h, 8)` shows `[1, 4, 2, 3, 3, 3, 3, 3]`: box 3 ended up pointing at itself, the cycle trap from "Where it goes wrong".
- Run `h = build_list([1, 2, 3, 2, 1]); print(is_palindrome_list(h), to_list(h))`: `True [1, 2, 3]`. The check rewired the input; reverse the back half again if the caller needs the list intact.
- In `is_palindrome_list`, change `while second:` to `while first:` and test `[1, 2, 2, 1]`: `AttributeError`. On even lengths the front half still reaches the shared middle box, one step after the reversed half has ended.

**Reverse in k-groups (25).** Each block is a small reversal. Two tricks make the glue easy: start the reversal with `prev` = the box *after* the block (so the block's new tail already points onward), and keep `group_prev`, the box before the block, to hook in the block's new head.

```text
 k = 2    D -> 1 -> 2 -> 3 -> 4 -> 5
          gp   ^    kth  nxt                 kth exists, so reverse the block 1..2
          D -> 2 -> 1 -> 3 -> 4 -> 5         prev started at nxt, so 1 already points at 3
                    gp                       the old first box (1) is the next group_prev
          D -> 2 -> 1 -> 4 -> 3 -> 5         next block; 5 alone (fewer than k) stays as it is
```

In [ ]:
def reverse_k_group(head, k):
    dummy = ListNode(0, head)
    group_prev = dummy                       # the box just before the current block
    while True:
        kth = group_prev                     # look ahead: are there k more boxes?
        for _ in range(k):
            kth = kth.next
            if kth is None:
                return dummy.next            # a short tail stays as it is
        nxt = kth.next                       # the first box after the block
        prev, cur = nxt, group_prev.next     # reverse the block; its new tail points to nxt
        while cur is not nxt:
            following = cur.next             # the same three moves as reverse_list
            cur.next = prev
            prev, cur = cur, following
        first = group_prev.next              # the old first box is now the block's tail
        group_prev.next = kth                # hook the block's new head in
        group_prev = first


print(to_list(reverse_k_group(build_list([1, 2, 3, 4, 5]), 2)))   # [2, 1, 4, 3, 5]
print(to_list(reverse_k_group(build_list([1, 2, 3, 4, 5]), 3)))   # [3, 2, 1, 4, 5]

**Try it**
- Start the block reversal with `prev = None` instead of `nxt` and rerun k = 2: `[2, 1]`. Box 1 now points at `None`, so everything after the first block is cut off.
- Predict before running: k = 1 returns the list unchanged, k = 5 reverses all of it, k = 6 changes nothing (there is no full block).
- Print `to_list(dummy.next)` right after `group_prev = first`: for k = 2 you see `[2, 1, 3, 4, 5]` and then `[2, 1, 4, 3, 5]`, one finished block per line.

**Two lists in lockstep (2, 160).** One finger per list, moving together. Add Two Numbers is schoolbook addition: the lists store the ones digit first, so walk both, write `total % 10`, carry `total // 10`. Intersection lines up the two tails: after the merge box both lists are the *same* boxes, so skip the longer list's extra prefix and the fingers reach the merge box at the same moment.

```text
 A:      4 -> 1 \                 lengths 5 and 6: move B's finger 1 box ahead,
                 8 -> 4 -> 5      then step together; they meet at the shared 8
 B: 5 -> 6 -> 1 /                 (the two 1s hold equal values but are different boxes)
```

In [ ]:
def add_two_numbers(l1, l2):                 # ones digit first: 342 is stored as 2 -> 4 -> 3
    dummy = tail = ListNode()
    carry = 0
    while l1 or l2 or carry:                 # a leftover carry still makes a box
        total = carry
        if l1:
            total, l1 = total + l1.val, l1.next
        if l2:
            total, l2 = total + l2.val, l2.next
        carry, digit = divmod(total, 10)
        tail.next = ListNode(digit)          # RECORD: one new box per column
        tail = tail.next
    return dummy.next


def intersection_node(a, b):
    def length(node):
        n = 0
        while node:
            n, node = n + 1, node.next
        return n
    la, lb = length(a), length(b)
    for _ in range(la - lb):                 # skip the longer list's extra prefix
        a = a.next
    for _ in range(lb - la):                 # (only one of these two loops runs)
        b = b.next
    while a is not b:                        # same distance to the end: walk together
        a, b = a.next, b.next
    return a                                 # the shared box, or None


print(to_list(add_two_numbers(build_list([2, 4, 3]), build_list([5, 6, 4]))))   # [7, 0, 8]  (342 + 465 = 807)
print(to_list(add_two_numbers(build_list([9, 9]), build_list([1]))))             # [0, 0, 1]  (99 + 1 = 100)
shared = build_list([8, 4, 5])
a, b = ListNode(4, ListNode(1, shared)), ListNode(5, ListNode(6, ListNode(1, shared)))
print(intersection_node(a, b).val, intersection_node(build_list([1]), build_list([1])))   # 8 None

**Try it**
- Change `while l1 or l2 or carry:` to `while l1 or l2:` and rerun 99 + 1: `[0, 0]`. The final carry never got its box.
- In `intersection_node`, compare values (`while a.val != b.val:`) and rerun: the first value printed is 1, not 8 (two different boxes hold a 1; the lists only merge at 8), and the two separate `[1]` lists now "intersect" too.
- A way to line up without counting: `pa, pb = a, b`, then `while pa is not pb: pa = pa.next if pa else b; pb = pb.next if pb else a`. A finger that falls off its list jumps to the other list's head, so both walk A's own part + B's own part + the shared tail, and arrive at the merge box together. Check that it also gives 8 (and `None` for two separate lists).

**Copy a list with random arrows (138).** A deep copy needs a translator from "a box of the old list" to "its copy". Pass 1 creates a copy of every box and stores the pair in a dict; pass 2 wires each copy's `next` and `random` through that dict. (The O(1)-space version weaves each copy right after its original, `A -> A' -> B -> B'`, so "the copy of X" is simply `X.next`; then it unweaves the two lists.)

In [ ]:
class RandomNode:
    def __init__(self, val, next=None, random=None):
        self.val, self.next, self.random = val, next, random


def copy_random_list(head):
    old_to_new = {None: None}                # "points at nothing" translates to nothing
    node = head
    while node:                              # pass 1: one new box per old box
        old_to_new[node] = RandomNode(node.val)
        node = node.next
    node = head
    while node:                              # pass 2: wire the copies through the dict
        old_to_new[node].next = old_to_new[node.next]
        old_to_new[node].random = old_to_new[node.random]
        node = node.next
    return old_to_new[head]


def build_random(pairs):                     # [(val, index the random arrow points to), ...]
    nodes = [RandomNode(v) for v, _ in pairs]
    for i, (_, r) in enumerate(pairs):
        nodes[i].next = nodes[i + 1] if i + 1 < len(nodes) else None
        nodes[i].random = nodes[r] if r is not None else None
    return nodes[0] if nodes else None


def describe(head):                          # [(val, index of the random target), ...]
    boxes = []
    while head:
        boxes.append(head)
        head = head.next
    return [(n.val, None if n.random is None else boxes.index(n.random)) for n in boxes]


orig = build_random([(7, None), (13, 0), (11, 4), (10, 2), (1, 0)])
clone = copy_random_list(orig)
print(describe(clone))                       # [(7, None), (13, 0), (11, 4), (10, 2), (1, 0)]
print(clone is orig, clone.next.random is orig, clone.next.random is clone)   # False False True

**Try it**
- Remove the `None: None` entry and rerun: `KeyError: None`, from the first box whose `random` points at nothing.
- In pass 2, write `old_to_new[node].random = node.random` (no translation): `describe(clone)` now fails with `ValueError`, because the copy's random arrows point into the *old* list. A deep copy must never point outside itself.
- Print `len(old_to_new)` after the copy: 6, the five boxes plus the `None` entry.

### Say it in the interview

> "Copying the values into an array would make this easy, but it costs O(n) extra space, so I'll rewire the nodes in place. I'll put a dummy node before the head so removing or inserting at the front isn't a special case. In the loop I save `cur.next` before I change it, then flip the pointer, then advance. Every node is touched once: O(n) time, O(1) extra space."

While coding, point at the save line ("so I don't lose the rest"), at `return dummy.next` ("the head may have changed"), and for fast/slow say the loop guard out loud ("fast can take two more steps"). Before claiming it works, draw three boxes and step through them on the whiteboard.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Add Two Numbers | `linked_list/add_two_numbers.py` | ones digit first, so add column by column with a carry; loop while either list or the carry remains |
| Copy List with Random Pointer | `linked_list/copy_list_with_random_pointer.py` | dict old box → new box (or weave copies after originals), then wire `next`/`random` through it |
| Find the Duplicate Number | `linked_list/find_the_duplicate_number.py` | read `i → nums[i]` as a list from index 0; the duplicate has two incoming arrows = the cycle entrance (Floyd) |
| Intersection of Two Linked Lists | `linked_list/intersection_of_two_linked_lists.py` | skip the longer list's extra prefix, then walk both until `a is b` |
| Linked List Cycle | `linked_list/linked_list_cycle.py` | fast gains one box per step on slow inside a loop; meeting = cycle, falling off = none |
| Linked List Cycle II | `linked_list/linked_list_cycle_ii.py` · `practice/simple/23_linked_list_cycle_ii.py` | after the meeting, head → entrance equals meeting point → entrance (mod the loop): restart one finger at the head |
| Merge Two Sorted Lists | `linked_list/merge_two_sorted_lists.py` · `practice/simple/22_merge_two_sorted_lists.py` | dummy + tail: always hang the smaller front box; attach the leftover whole |
| Palindrome Linked List | `linked_list/palindrome_linked_list.py` | find the middle, reverse the back half in place, compare the halves |
| Remove Nth Node From End of List | `linked_list/remove_nth_from_end.py` · `practice/simple/25_remove_nth_node_from_end.py` | gap of n + 1 from a dummy: when fast falls off, slow is just before the victim |
| Reorder List | `linked_list/reorder_list.py` | middle, cut, reverse the back half, zip the two halves |
| Reverse Linked List | `linked_list/reverse_linked_list.py` · `practice/simple/21_reverse_linked_list.py` | save next, flip the arrow, advance; `prev` ends as the new head |
| Reverse Nodes in k-Group | `linked_list/reverse_nodes_in_k_group.py` | look ahead k boxes; reverse with `prev` = box after the block; hook `group_prev` to the k-th box |

### Self-check

1. Why does the reversal loop need three fingers (`prev`, `cur`, `nxt`) and not two?
<details><summary>Answer</summary><code>prev</code> is what <code>cur</code> must point back to, and <code>cur</code> is the box being flipped. The moment <code>cur.next</code> is overwritten, the rest of the list is unreachable unless a third finger already holds it, so <code>nxt</code> must be saved first.</details>

2. In Linked List Cycle II, why does a finger restarted at the head meet the other finger exactly at the entrance?
<details><summary>Answer</summary>With a tail of a boxes, a loop of c boxes and a meeting point b boxes into the loop, slow walked a + b and fast 2(a + b). Fast's extra a + b steps were whole laps, so a + b = k·c and a = (c − b) + (k − 1)·c. Walking a steps from the meeting point therefore lands on the entrance, the same place a steps from the head reach.</details>

3. Why does `remove_nth_from_end` start both fingers on the dummy and open a gap of n + 1 rather than n?
<details><summary>Answer</summary>To delete a box you need the box <em>before</em> it. A gap of n + 1 leaves <code>slow</code> one box before the victim when <code>fast</code> falls off. Starting from the dummy makes that work even when the victim is the head, because the dummy is the head's "box before".</details>

4. In Find the Duplicate Number, why can index 0 never be inside the cycle, and why is the cycle's entrance the duplicate?
<details><summary>Answer</summary>Every value is in [1, n], so no arrow points at index 0: the walk leaves it and never returns, so it is the start of the tail. The entrance is the one box with two incoming arrows (one from the tail, one from inside the loop), and two arrows into box v means two indices hold the value v.</details>